# Mini-projet : PageRank d'un mini-web

| Module | Durée | Prérequis |
|---|---|---|
| M7 – Valeurs propres et dynamique | 150 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

L'association cycliste Roue Libre veut proposer sur son site un petit moteur de recherche consacré au vélo en ville. Elle a recensé 20 pages web de la ville qui parlent de vélo : site de la mairie, carte des pistes, blogs, forum, magasins, location, etc. Les fichiers `donnees/pages.csv` et `donnees/liens.csv` décrivent ces pages et les liens de l'une vers l'autre. Ce mini-web est fictif, mais construit comme un vrai.

Quand plusieurs pages répondent à une recherche, dans quel ordre les afficher ? En 1998, les fondateurs de Google ont proposé une réponse devenue célèbre, le **PageRank** : une page est importante si des pages importantes pointent vers elle.

**Les questions de l'association : quel classement le PageRank donne-t-il pour ces 20 pages ? Diffère-t-il du simple comptage des liens reçus ? À quel point dépend-il de ses réglages, et peut-on le manipuler ?**

**Activité** : « Une page est importante si des pages importantes pointent vers elle. » Cette définition tourne en rond. Pourquoi n'est-ce pas un problème, d'après ce que vous avez vu dans ce module ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pages = pd.read_csv("donnees/pages.csv")
liens = pd.read_csv("donnees/liens.csv")
print(f"{len(pages)} pages, {len(liens)} liens")
print("\nLiens reçus par page :")
print(liens["cible"].value_counts().reindex(pages["page"], fill_value=0).sort_values(ascending=False).to_string())

In [ ]:
pari_premiere_page = "carte-pistes"     # la page qui sera classée première (modifiez)

## ③ Abstraire : le surfeur aléatoire

Imaginons un internaute qui surfe au hasard. Sur chaque page, il clique sur l'un des liens, choisi au hasard. De temps en temps, il s'ennuie et saute vers une page quelconque. Le PageRank d'une page est la **proportion du temps** qu'il y passe à long terme.

![Schéma : quatre pages reliées par des liens ; le surfeur suit un lien avec la probabilité d ou saute au hasard ; le PageRank est le vecteur propre de G pour la valeur propre 1](../../assets/schemas/schema-surfeur.svg)

C'est une **chaîne de Markov** (notebook 7.3) dont les états sont les pages :

- la **matrice des liens** $M$ : si la page $j$ contient $k_j$ liens, chacun est suivi avec la probabilité $1/k_j$, donc $M_{ij} = 1/k_j$ si $j$ pointe vers $i$ ;
- une page **sans lien sortant** (un document PDF, par exemple) piégerait le surfeur : on considère qu'il en repart vers n'importe quelle page, avec la probabilité $1/n$ ;
- le **facteur d'amortissement** $d$ (classiquement 0,85) est la probabilité de suivre un lien plutôt que de sauter au hasard.

$$
G = d\,M + \frac{1 - d}{n}\,J,
\qquad
\mathbf{r} = G\,\mathbf{r},
$$

où $J$ est la matrice remplie de 1. Tous les coefficients de $G$ sont strictement positifs : d'après le notebook 7.3, l'état stationnaire $\mathbf{r}$ existe, il est unique, et la méthode de la puissance (notebook 7.4) le trouve.

**Activité** : Pour les quatre pages du schéma, avec $d = 1$, écrivez la matrice $M$. Vérifiez que chaque colonne a pour somme 1.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Construire la matrice de Google

In [ ]:
noms = list(pages["page"])
indice = {p: k for k, p in enumerate(noms)}
n = len(noms)

def matrice_liens(liens, noms):
    n = len(noms)
    indice = {p: k for k, p in enumerate(noms)}
    M = np.zeros((n, n))
    for source, cible in zip(liens["source"], liens["cible"]):
        M[indice[cible], indice[source]] = 1
    sortants = M.sum(axis=0)
    for j in range(n):
        M[:, j] = M[:, j] / sortants[j] if sortants[j] > 0 else 1 / n    # page sans lien : saut au hasard
    return M

def matrice_google(M, d=0.85):
    return d * M + (1 - d) / len(M) * np.ones_like(M)

def puissance(G, tol=1e-12, max_iter=100_000):
    r = np.ones(len(G)) / len(G)
    for k in range(1, max_iter + 1):
        nouveau = G @ r
        nouveau /= nouveau.sum()          # on garde une somme de 1 : ce sont des proportions
        if np.abs(nouveau - r).sum() < tol:
            return nouveau, k
        r = nouveau
    return r, max_iter

M = matrice_liens(liens, noms)
G = matrice_google(M)
print("Colonnes de somme 1 ?", np.allclose(G.sum(axis=0), 1), "  coefficients tous positifs ?", (G > 0).all())

### Le classement

In [ ]:
r, iterations = puissance(G)
recus = liens["cible"].value_counts().reindex(noms, fill_value=0)
classement = pd.DataFrame({"titre": pages["titre"].values, "PageRank": r, "liens reçus": recus.values}, index=noms)
classement = classement.sort_values("PageRank", ascending=False)
classement["rang"] = range(1, n + 1)
classement["rang selon les liens reçus"] = classement["liens reçus"].rank(ascending=False, method="min").astype(int)
print(f"Convergence en {iterations} itérations\n")
print(classement[["PageRank", "rang", "liens reçus", "rang selon les liens reçus"]].round(4).to_string())

**Activité** : Repérez deux pages dont le rang PageRank diffère nettement du rang selon les liens reçus. Expliquez pourquoi, en regardant **qui** pointe vers elles.

Indiquez vos réponses et justifiez-les.

### Corroborer : simuler le surfeur

In [ ]:
rng = np.random.default_rng(42)
visites = np.zeros(n)
page = 0
for _ in range(200_000):
    page = rng.choice(n, p=G[:, page])
    visites[page] += 1
frequences = visites / visites.sum()
print(f"Écart maximal entre fréquences simulées et PageRank : {np.abs(frequences - r).max():.4f}")

### Le facteur d'amortissement, un paramètre

In [ ]:
facteurs = [0.5, 0.7, 0.85, 0.95, 0.99]
resultats = {}
for d in facteurs:
    rd, it = puissance(matrice_google(M, d))
    resultats[d] = rd
    top = [noms[k] for k in np.argsort(-rd)[:3]]
    print(f"d = {d:.2f} : {it:5d} itérations   podium : {', '.join(top)}")

fig, ax = plt.subplots(figsize=(10, 4))
ordre = np.argsort(-r)
for d, couleur in zip(facteurs, plt.colormaps["liouai"](np.linspace(0.3, 1, len(facteurs)))):
    ax.plot(range(n), resultats[d][ordre], "o-", ms=4, color=couleur, label=f"d = {d}")
ax.set_xticks(range(n), [noms[k] for k in ordre], rotation=75, fontsize=8)
ax.set_ylabel("PageRank"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("PageRank de chaque page selon le facteur d'amortissement")
plt.tight_layout(); plt.show()

**Activité** : Comment le nombre d'itérations évolue-t-il avec $d$ ? Reliez-le au notebook 7.4 : on peut montrer que $\lvert \lambda_2 \rvert \leq d$ pour la matrice de Google. Que deviennent les écarts entre pages quand $d$ diminue ? Pourquoi ?

Indiquez vos réponses et justifiez-les.

### Peut-on tricher ?

Le gérant de Magasin Promo-Vélo crée six pages sans intérêt qui pointent toutes vers sa page, et sa page pointe vers elles. C'est une **ferme de liens**.

In [ ]:
nouvelles = [f"promo-{k}" for k in range(1, 7)]
liens_triches = pd.concat([liens,
                           pd.DataFrame({"source": nouvelles, "cible": "magasin-promo"}),
                           pd.DataFrame({"source": "magasin-promo", "cible": nouvelles})], ignore_index=True)
noms_triches = noms + nouvelles
r_triche, _ = puissance(matrice_google(matrice_liens(liens_triches, noms_triches)))

rang_avant = list(np.array(noms)[np.argsort(-r)]).index("magasin-promo") + 1
rang_apres = list(np.array(noms_triches)[np.argsort(-r_triche)]).index("magasin-promo") + 1
print(f"magasin-promo : rang {rang_avant} sur {n} avant, rang {rang_apres} sur {len(noms_triches)} après la ferme de liens")
print(f"PageRank : {r[indice['magasin-promo']]:.4f} avant, {r_triche[noms_triches.index('magasin-promo')]:.4f} après")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_premiere_page} ; première page : {classement.index[0]}")

**Activité** :

1. Quelles sont les trois pages les plus importantes ? Le résultat vous paraît-il raisonnable pour un internaute qui cherche des informations sur le vélo en ville ?
2. Le classement est-il robuste au choix de $d$ ? Les trois premières places changent-elles ?
3. Quelles hypothèses sur le comportement des internautes le modèle du surfeur aléatoire fait-il ? Lesquelles sont discutables ? Quel effet auraient-elles sur le classement ?
4. La ferme de liens fonctionne-t-elle ? Quelles parades proposeriez-vous ?

Indiquez vos réponses et justifiez-les.

### Note à l'association

Rédigez une note d'une demi-page, sans formule, qui explique :

- le principe du classement, avec l'image du surfeur aléatoire ;
- le classement obtenu et en quoi il diffère d'un simple comptage des liens ;
- les réglages et leur influence ;
- les limites : hypothèses du modèle, possibilité de manipulation.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Que vaut le classement ?** Le PageRank mesure l'importance d'une page dans le réseau des liens, pas la qualité de son contenu ni sa pertinence pour une recherche donnée. Comment combiner les deux ?
- **Et sans saut au hasard ?** Essayez $d = 1$. La méthode converge-t-elle ? Le résultat a-t-il un sens ? Que se passerait-il si l'on ne traitait pas à part la page sans lien sortant ?
- **Généraliser : un PageRank personnalisé.** Au lieu de sauter vers **n'importe quelle** page, le surfeur pourrait sauter seulement vers des pages d'un thème qui l'intéresse. Écrivez une fonction `pagerank_personnalise(M, preferences, d=0.85)`, où `preferences` est un vecteur de probabilités de saut (de somme 1). Calculez le classement pour un internaute qui s'intéresse à la **sécurité** (sauts vers `blog-securite`, `ecole-velo` et `forum-vol`). Comment le podium change-t-il ?

In [ ]:
def pagerank_personnalise(M, preferences, d=0.85):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def pagerank_personnalise(M, preferences, d=0.85):
    preferences = np.asarray(preferences, dtype=float)
    G_perso = d * M + (1 - d) * np.outer(preferences, np.ones(len(M)))   # colonne j : d M[:, j] + (1-d) preferences
    return puissance(G_perso)[0]

securite = np.zeros(n)
for p in ["blog-securite", "ecole-velo", "forum-vol"]:
    securite[indice[p]] = 1 / 3
r_securite = pagerank_personnalise(M, securite)
print("Podium général  :", [noms[k] for k in np.argsort(-r)[:5]])
print("Podium sécurité :", [noms[k] for k in np.argsort(-r_securite)[:5]])

r1, it1 = puissance(M, max_iter=5000)
print(f"\nAvec d = 1 : {it1} itérations ; pages les moins visitées :",
      ", ".join(f"{noms[k]} ({r1[k]:.4f})" for k in np.argsort(r1)[:3]))

M_brute = np.zeros((n, n))
for source, cible in zip(liens["source"], liens["cible"]):
    M_brute[indice[cible], indice[source]] = 1
sortants = M_brute.sum(axis=0)
M_brute[:, sortants > 0] /= sortants[sortants > 0]          # la colonne de pdf-plan-velo reste nulle
p = np.ones(n) / n
for t in range(50):
    p = M_brute @ p
print(f"Sans traitement de la page sans lien, la probabilité totale après 50 clics vaut {p.sum():.4f} : "
      "le surfeur « disparaît » peu à peu dans le document PDF.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Transformer des constantes en paramètres": 1,
    "Repérer les traits intéressants du résultat": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Expliquer par écrit": 1,
    "Sentir la fiabilité d'un modèle": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M7-projet-pagerank", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")